In [54]:
import pandas as pd
import re

with open("sample_logs.txt", "r") as f:
    all_lines = f.readlines()

total_lines = len(all_lines)
non_empty_logs = [line.strip() for line in all_lines if line.strip()]
empty_lines = [line for line in all_lines if not line.strip()]

print("Total lines in file:", total_lines)
print("Non-empty logs:", len(non_empty_logs))
print("Empty/blank lines:", len(empty_lines))

df = pd.DataFrame({"raw_log": non_empty_logs})

# Basic cleaning
df['raw_log'] = df['raw_log'].str.lower()

display(df.head(10))

Total lines in file: 5200
Non-empty logs: 5195
Empty/blank lines: 5


,raw_log
0,mar 15 06:00:03 prod-app-01 api-gateway[3330]: <info> health check passed upstream=auth-service
1,2026-03-15 06:00:03 info [api-gateway] request completed method=get path=/api/v2/users/3591 status=200 duration=264ms
2,2026-03-15 06:00:07 warn [inventory] sync delay from erp: last update 47 minutes ago
3,2026-03-15 06:00:09 info [inventory] stock updated sku=gw-7547 qty=328 warehouse=wh-03
4,2026-03-15t06:00:11.927231z error auth-service: mfa verification failed uid=2160 code=invalid
5,2026-03-15 06:00:14 warn [api-gateway] request slow method=get path=/api/v2/search status=200 duration=2021ms
6,[06:00:17] i/auth-service: password reset requested uid=9085
7,mar 15 06:00:21 prod-app-01 inventory[4727]: <info> inventory sync completed source=erp records=2847 duration=12.4s
8,[06:00:25] i/email-service: email sent to=user677@company.io template=welcome msg_id=msg-739098
9,[06:00:28] i/api-gateway: request completed method=get path=/api/v2/products status=200 duration=49ms


In [55]:
import re

def normalize_log(log):
    log = log.lower()
    
    # Replace IP addresses
    log = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', log)
    
    # Replace URLs/paths
    log = re.sub(r'\/[^\s]+', '<PATH>', log)
    
    # Replace timestamps (common patterns)
    log = re.sub(r'\b\d{2}:\d{2}:\d{2}\b', '<TIME>', log)
    
    # Replace standalone numbers (but preserve meaning)
    log = re.sub(r'\b\d+\b', '<NUM>', log)
    
    # Clean extra spaces
    log = re.sub(r'\s+', ' ', log).strip()
    
    return log

df["normalized_log"] = df["raw_log"].apply(normalize_log)

print("Normalized logs preview:")
display(df[["raw_log", "normalized_log"]].head(10))

Normalized logs preview:


,raw_log,normalized_log
0,mar 15 06:00:03 prod-app-01 api-gateway[3330]: <info> health check passed upstream=auth-service,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service
1,2026-03-15 06:00:03 info [api-gateway] request completed method=get path=/api/v2/users/3591 status=200 duration=264ms,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms
2,2026-03-15 06:00:07 warn [inventory] sync delay from erp: last update 47 minutes ago,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago
3,2026-03-15 06:00:09 info [inventory] stock updated sku=gw-7547 qty=328 warehouse=wh-03,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>
4,2026-03-15t06:00:11.927231z error auth-service: mfa verification failed uid=2160 code=invalid,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid
5,2026-03-15 06:00:14 warn [api-gateway] request slow method=get path=/api/v2/search status=200 duration=2021ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] request slow method=get path=<PATH> status=<NUM> duration=2021ms
6,[06:00:17] i/auth-service: password reset requested uid=9085,[<TIME>] i<PATH> password reset requested uid=<NUM>
7,mar 15 06:00:21 prod-app-01 inventory[4727]: <info> inventory sync completed source=erp records=2847 duration=12.4s,mar <NUM> <TIME> prod-app-<NUM> inventory[<NUM>]: <info> inventory sync completed source=erp records=<NUM> duration=<NUM>.4s
8,[06:00:25] i/email-service: email sent to=user677@company.io template=welcome msg_id=msg-739098,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>
9,[06:00:28] i/api-gateway: request completed method=get path=/api/v2/products status=200 duration=49ms,[<TIME>] i<PATH> request completed method=get path=<PATH> status=<NUM> duration=49ms


In [56]:
def generate_template(log):
    words = log.split()
    
    template = []
    for w in words:
        # Replace typed placeholders with generic wildcard
        if w in ['<ip>', '<path>', '<time>', '<num>']:
            template.append('*')
        else:
            template.append(w)
    
    return " ".join(template)

df["template"] = df["normalized_log"].apply(generate_template)

print("Template preview:")
display(df[["normalized_log", "template"]].head(10))

Template preview:


,normalized_log,template
0,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service
1,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms
2,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago
3,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>
4,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid
5,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] request slow method=get path=<PATH> status=<NUM> duration=2021ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] request slow method=get path=<PATH> status=<NUM> duration=2021ms
6,[<TIME>] i<PATH> password reset requested uid=<NUM>,[<TIME>] i<PATH> password reset requested uid=<NUM>
7,mar <NUM> <TIME> prod-app-<NUM> inventory[<NUM>]: <info> inventory sync completed source=erp records=<NUM> duration=<NUM>.4s,mar <NUM> <TIME> prod-app-<NUM> inventory[<NUM>]: <info> inventory sync completed source=erp records=<NUM> duration=<NUM>.4s
8,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>
9,[<TIME>] i<PATH> request completed method=get path=<PATH> status=<NUM> duration=49ms,[<TIME>] i<PATH> request completed method=get path=<PATH> status=<NUM> duration=49ms


In [57]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Step 1: Unique templates
unique_templates = df["template"].unique().tolist()

# Step 2: Generate embeddings for templates
model = SentenceTransformer('all-MiniLM-L6-v2')
template_embeddings = model.encode(unique_templates, show_progress_bar=True)

# Step 3: Compute similarity matrix
similarity_matrix = cosine_similarity(template_embeddings)

# Step 4: Merge similar templates based on threshold
SIM_THRESHOLD = 0.85

parent = list(range(len(unique_templates)))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[py] = px

for i in range(len(unique_templates)):
    for j in range(i + 1, len(unique_templates)):
        if similarity_matrix[i][j] >= SIM_THRESHOLD:
            union(i, j)

# Step 5: Assign Event IDs after merging
cluster_map = {}
event_counter = 1

template_to_event = {}

for i, template in enumerate(unique_templates):
    root = find(i)
    
    if root not in cluster_map:
        cluster_map[root] = f"E{event_counter:04d}"
        event_counter += 1
    
    template_to_event[template] = cluster_map[root]

# Step 6: Map back to dataframe
df["event_id"] = df["template"].map(template_to_event)

print("Total unique Event IDs after semantic merge:", len(set(template_to_event.values())))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Total unique Event IDs after semantic merge: 99


In [58]:
df_final = df[["event_id", "raw_log", "normalized_log", "template"]]

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("\nFirst 50 structured logs:")
display(df_final.head(50))

print("\nLast 50 structured logs:")
display(df_final.tail(50))


First 50 structured logs:


,event_id,raw_log,normalized_log,template
0,E0001,mar 15 06:00:03 prod-app-01 api-gateway[3330]: <info> health check passed upstream=auth-service,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service
1,E0002,2026-03-15 06:00:03 info [api-gateway] request completed method=get path=/api/v2/users/3591 status=200 duration=264ms,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms
2,E0003,2026-03-15 06:00:07 warn [inventory] sync delay from erp: last update 47 minutes ago,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago
3,E0004,2026-03-15 06:00:09 info [inventory] stock updated sku=gw-7547 qty=328 warehouse=wh-03,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>
4,E0005,2026-03-15t06:00:11.927231z error auth-service: mfa verification failed uid=2160 code=invalid,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid
5,E0002,2026-03-15 06:00:14 warn [api-gateway] request slow method=get path=/api/v2/search status=200 duration=2021ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] request slow method=get path=<PATH> status=<NUM> duration=2021ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] request slow method=get path=<PATH> status=<NUM> duration=2021ms
6,E0006,[06:00:17] i/auth-service: password reset requested uid=9085,[<TIME>] i<PATH> password reset requested uid=<NUM>,[<TIME>] i<PATH> password reset requested uid=<NUM>
7,E0003,mar 15 06:00:21 prod-app-01 inventory[4727]: <info> inventory sync completed source=erp records=2847 duration=12.4s,mar <NUM> <TIME> prod-app-<NUM> inventory[<NUM>]: <info> inventory sync completed source=erp records=<NUM> duration=<NUM>.4s,mar <NUM> <TIME> prod-app-<NUM> inventory[<NUM>]: <info> inventory sync completed source=erp records=<NUM> duration=<NUM>.4s
8,E0007,[06:00:25] i/email-service: email sent to=user677@company.io template=welcome msg_id=msg-739098,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>
9,E0008,[06:00:28] i/api-gateway: request completed method=get path=/api/v2/products status=200 duration=49ms,[<TIME>] i<PATH> request completed method=get path=<PATH> status=<NUM> duration=49ms,[<TIME>] i<PATH> request completed method=get path=<PATH> status=<NUM> duration=49ms



Last 50 structured logs:


,event_id,raw_log,normalized_log,template
5145,E0002,2026-03-15t08:54:41.945573z info api-gateway: request completed method=get path=/api/v2/users/3621 status=200 duration=33ms,<NUM>-<NUM>-15t08:<NUM>:<NUM>.945573z info api-gateway: request completed method=get path=<PATH> status=<NUM> duration=33ms,<NUM>-<NUM>-15t08:<NUM>:<NUM>.945573z info api-gateway: request completed method=get path=<PATH> status=<NUM> duration=33ms
5146,E0058,"{""ts"": ""2026-03-15t08:54:45.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-20266194""}","{""ts"": ""<NUM>-<NUM>-15t08:<NUM>:<NUM>.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-<NUM>""}","{""ts"": ""<NUM>-<NUM>-15t08:<NUM>:<NUM>.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-<NUM>""}"
5147,E0012,"{""ts"": ""2026-03-15t08:54:46.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=8962""}","{""ts"": ""<NUM>-<NUM>-15t08:<NUM>:<NUM>.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=<NUM>""}","{""ts"": ""<NUM>-<NUM>-15t08:<NUM>:<NUM>.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=<NUM>""}"
5148,E0026,mar 15 08:54:49 prod-app-01 payment-svc[3749]: <info> invoice generated invoice_id=inv-20264112,mar <NUM> <TIME> prod-app-<NUM> payment-svc[<NUM>]: <info> invoice generated invoice_id=inv-<NUM>,mar <NUM> <TIME> prod-app-<NUM> payment-svc[<NUM>]: <info> invoice generated invoice_id=inv-<NUM>
5149,E0044,mar 15 08:54:50 prod-web-01 auth-service[9287]: <warn> rate limit approaching for ip 10.0.1.130,mar <NUM> <TIME> prod-web-<NUM> auth-service[<NUM>]: <warn> rate limit approaching for ip <IP>,mar <NUM> <TIME> prod-web-<NUM> auth-service[<NUM>]: <warn> rate limit approaching for ip <IP>
5150,E0019,2026-03-15 08:54:52 warn [api-gateway] upstream timeout upstream=inventory duration=5000ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] upstream timeout upstream=inventory duration=5000ms,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] upstream timeout upstream=inventory duration=5000ms
5151,E0003,2026-03-15 08:54:57 info [inventory] inventory sync completed source=erp records=2847 duration=12.4s,<NUM>-<NUM>-<NUM> <TIME> info [inventory] inventory sync completed source=erp records=<NUM> duration=<NUM>.4s,<NUM>-<NUM>-<NUM> <TIME> info [inventory] inventory sync completed source=erp records=<NUM> duration=<NUM>.4s
5152,E0005,2026-03-15 08:55:00 info [auth-service] user login successful uid=1916,<NUM>-<NUM>-<NUM> <TIME> info [auth-service] user login successful uid=<NUM>,<NUM>-<NUM>-<NUM> <TIME> info [auth-service] user login successful uid=<NUM>
5153,E0002,mar 15 08:55:02 prod-web-01 api-gateway[7557]: <warn> rate limit exceeded client_id=app-908 limit=1000/min,mar <NUM> <TIME> prod-web-<NUM> api-gateway[<NUM>]: <warn> rate limit exceeded client_id=app-<NUM> limit=<NUM><PATH>,mar <NUM> <TIME> prod-web-<NUM> api-gateway[<NUM>]: <warn> rate limit exceeded client_id=app-<NUM> limit=<NUM><PATH>
5154,E0005,15/mar/2026:08:55:02 +0530 | warn | auth-service | login attempt with expired token uid=8744,<NUM><PATH> +<NUM> | warn | auth-service | login attempt with expired token uid=<NUM>,<NUM><PATH> +<NUM> | warn | auth-service | login attempt with expired token uid=<NUM>


In [59]:
# Create event summary with template meaning
event_summary = df.groupby("event_id").agg({
    "template": "first",
    "event_id": "count"
}).rename(columns={"event_id": "count"}).reset_index()

total_event_ids = len(event_summary)

print("Total unique Event IDs:", total_event_ids)

print("\nTop Event IDs:")
display(event_summary.sort_values(by="count", ascending=False).head(100))

print(f"Total logs: {len(df)}")
print(f"Total unique Event IDs: {total_event_ids}")
print(f"Average logs per Event ID: {len(df) // total_event_ids}")

Total unique Event IDs: 99

Top Event IDs:


,event_id,template,count
4,E0005,<NUM>-<NUM>-15t06:<NUM>:<NUM>.927231z error auth-service: mfa verification failed uid=<NUM> code=invalid,710
1,E0002,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] request completed method=get path=<PATH> status=<NUM> duration=264ms,500
8,E0009,"{""ts"": ""<NUM>-<NUM>-15t06:<NUM>:<NUM>.047373z"", ""level"": ""info"", ""svc"": ""api-gateway"", ""msg"": ""health check passed upstream=inventory""}",311
0,E0001,mar <NUM> <TIME> prod-app-<NUM> api-gateway[<NUM>]: <info> health check passed upstream=auth-service,305
6,E0007,[<TIME>] i<PATH> email sent to=user677@company.io template=welcome msg_id=msg-<NUM>,192
13,E0014,<NUM>-<NUM>-<NUM> <TIME> warn [scheduler] cron job slow job=daily_report_generation duration=<NUM>.3s threshold=60s,144
2,E0003,<NUM>-<NUM>-<NUM> <TIME> warn [inventory] sync delay from erp: last update <NUM> minutes ago,134
11,E0012,"{""ts"": ""<NUM>-<NUM>-15t06:<NUM>:<NUM>.842791z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""user logout uid=<NUM>""}",131
14,E0015,"{""ts"": ""<NUM>-<NUM>-15t06:<NUM>:<NUM>.501009z"", ""level"": ""info"", ""svc"": ""inventory"", ""msg"": ""stock updated sku=cw-<NUM> qty=<NUM> warehouse=wh-<NUM>"", ""trace_id"": ""trace-<NUM>""}",131
3,E0004,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=gw-<NUM> qty=<NUM> warehouse=wh-<NUM>,124


Total logs: 5195
Total unique Event IDs: 99
Average logs per Event ID: 52


In [61]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

from IPython.display import display, HTML

display(HTML("""
<style>
.dataframe {
    max-height: 500px;
    overflow-y: scroll;
    display: block;
}
</style>
"""))

# 🔥 Only first 50 rows (clean + fast)
display(df_final.head(50).style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
}))

# 🔥 Optional: last 50 for comparison
display(df_final.tail(50).style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
}))

,event_id,raw_log,normalized_log,template
0,E0001,mar 15 06:00:03 prod-app-01 api-gateway[3330]: health check passed upstream=auth-service,mar prod-app- api-gateway[]: health check passed upstream=auth-service,mar prod-app- api-gateway[]: health check passed upstream=auth-service
1,E0002,2026-03-15 06:00:03 info [api-gateway] request completed method=get path=/api/v2/users/3591 status=200 duration=264ms,-- info [api-gateway] request completed method=get path= status= duration=264ms,-- info [api-gateway] request completed method=get path= status= duration=264ms
2,E0003,2026-03-15 06:00:07 warn [inventory] sync delay from erp: last update 47 minutes ago,-- warn [inventory] sync delay from erp: last update minutes ago,-- warn [inventory] sync delay from erp: last update minutes ago
3,E0004,2026-03-15 06:00:09 info [inventory] stock updated sku=gw-7547 qty=328 warehouse=wh-03,-- info [inventory] stock updated sku=gw- qty= warehouse=wh-,-- info [inventory] stock updated sku=gw- qty= warehouse=wh-
4,E0005,2026-03-15t06:00:11.927231z error auth-service: mfa verification failed uid=2160 code=invalid,--15t06::.927231z error auth-service: mfa verification failed uid= code=invalid,--15t06::.927231z error auth-service: mfa verification failed uid= code=invalid
5,E0002,2026-03-15 06:00:14 warn [api-gateway] request slow method=get path=/api/v2/search status=200 duration=2021ms,-- warn [api-gateway] request slow method=get path= status= duration=2021ms,-- warn [api-gateway] request slow method=get path= status= duration=2021ms
6,E0006,[06:00:17] i/auth-service: password reset requested uid=9085,[] i password reset requested uid=,[] i password reset requested uid=
7,E0003,mar 15 06:00:21 prod-app-01 inventory[4727]: inventory sync completed source=erp records=2847 duration=12.4s,mar prod-app- inventory[]: inventory sync completed source=erp records= duration=.4s,mar prod-app- inventory[]: inventory sync completed source=erp records= duration=.4s
8,E0007,[06:00:25] i/email-service: email sent to=user677@company.io template=welcome msg_id=msg-739098,[] i email sent to=user677@company.io template=welcome msg_id=msg-,[] i email sent to=user677@company.io template=welcome msg_id=msg-
9,E0008,[06:00:28] i/api-gateway: request completed method=get path=/api/v2/products status=200 duration=49ms,[] i request completed method=get path= status= duration=49ms,[] i request completed method=get path= status= duration=49ms


,event_id,raw_log,normalized_log,template
5145,E0002,2026-03-15t08:54:41.945573z info api-gateway: request completed method=get path=/api/v2/users/3621 status=200 duration=33ms,--15t08::.945573z info api-gateway: request completed method=get path= status= duration=33ms,--15t08::.945573z info api-gateway: request completed method=get path= status= duration=33ms
5146,E0058,"{""ts"": ""2026-03-15t08:54:45.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-20266194""}","{""ts"": ""--15t08::.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-""}","{""ts"": ""--15t08::.696086z"", ""level"": ""info"", ""svc"": ""payment-svc"", ""msg"": ""invoice generated invoice_id=inv-""}"
5147,E0012,"{""ts"": ""2026-03-15t08:54:46.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=8962""}","{""ts"": ""--15t08::.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=""}","{""ts"": ""--15t08::.228665z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""mfa verification passed uid=""}"
5148,E0026,mar 15 08:54:49 prod-app-01 payment-svc[3749]: invoice generated invoice_id=inv-20264112,mar prod-app- payment-svc[]: invoice generated invoice_id=inv-,mar prod-app- payment-svc[]: invoice generated invoice_id=inv-
5149,E0044,mar 15 08:54:50 prod-web-01 auth-service[9287]: rate limit approaching for ip 10.0.1.130,mar prod-web- auth-service[]: rate limit approaching for ip,mar prod-web- auth-service[]: rate limit approaching for ip
5150,E0019,2026-03-15 08:54:52 warn [api-gateway] upstream timeout upstream=inventory duration=5000ms,-- warn [api-gateway] upstream timeout upstream=inventory duration=5000ms,-- warn [api-gateway] upstream timeout upstream=inventory duration=5000ms
5151,E0003,2026-03-15 08:54:57 info [inventory] inventory sync completed source=erp records=2847 duration=12.4s,-- info [inventory] inventory sync completed source=erp records= duration=.4s,-- info [inventory] inventory sync completed source=erp records= duration=.4s
5152,E0005,2026-03-15 08:55:00 info [auth-service] user login successful uid=1916,-- info [auth-service] user login successful uid=,-- info [auth-service] user login successful uid=
5153,E0002,mar 15 08:55:02 prod-web-01 api-gateway[7557]: rate limit exceeded client_id=app-908 limit=1000/min,mar prod-web- api-gateway[]: rate limit exceeded client_id=app- limit=,mar prod-web- api-gateway[]: rate limit exceeded client_id=app- limit=
5154,E0005,15/mar/2026:08:55:02 +0530 | warn | auth-service | login attempt with expired token uid=8744,+ | warn | auth-service | login attempt with expired token uid=,+ | warn | auth-service | login attempt with expired token uid=


In [62]:
# Count frequency of each Event ID
event_counts = df['event_id'].value_counts().reset_index()
event_counts.columns = ['event_id', 'count']

# ✅ Take only ONE template per event_id (most frequent one)
event_templates = df.groupby('event_id')['template'].agg(lambda x: x.mode()[0]).reset_index()

# Merge correctly
event_counts = event_counts.merge(event_templates, on='event_id', how='left')

# Reorder
event_counts = event_counts[['event_id', 'template', 'count']]

# Attach back to main df (safe way)
df = df.drop(columns=['count', 'rare_event'], errors='ignore')
count_dict = event_counts.set_index('event_id')['count'].to_dict()
df['count'] = df['event_id'].map(count_dict)

# Rare events
RARE_THRESHOLD = 5
df['rare_event'] = df['count'] < RARE_THRESHOLD

# Display clean output
print("Top Events:")
display(event_counts.sort_values(by='count', ascending=False).head(20))

print("\nRare Events:")
display(event_counts.sort_values(by='count', ascending=True).head(20))

Top Events:


,event_id,template,count
0,E0005,<NUM>-<NUM>-<NUM> <TIME> info [auth-service] token refreshed for uid=<NUM>,710
1,E0002,<NUM>-<NUM>-<NUM> <TIME> warn [api-gateway] rate limit exceeded client_id=app-<NUM> limit=<NUM><PATH>,500
2,E0009,<NUM>-<NUM>-<NUM> <TIME> error [api-gateway] request entity too large method=post path=<PATH> size=<NUM> limit=<NUM>,311
3,E0001,<NUM>-<NUM>-<NUM> <TIME> info [api-gateway] health check passed upstream=auth-service,305
4,E0007,<NUM>-<NUM>-<NUM> <TIME> info [email-service] email sent to=user187@company.io template=welcome msg_id=msg-<NUM>,192
5,E0014,<NUM>-<NUM>-<NUM> <TIME> warn [scheduler] cron job slow job=daily_report_generation duration=<NUM>.3s threshold=60s,144
6,E0003,<NUM>-<NUM>-<NUM> <TIME> info [inventory] inventory sync completed source=erp records=<NUM> duration=<NUM>.4s,134
7,E0012,"{""ts"": ""<NUM>-<NUM>-15t06:<NUM>:<NUM>.002701z"", ""level"": ""info"", ""svc"": ""auth-service"", ""msg"": ""user logout uid=<NUM>""}",131
8,E0015,<NUM><PATH> +<NUM> | error | inventory | failed to reserve stock sku=dz-<NUM> requested=<NUM> available=<NUM>,131
9,E0004,<NUM>-<NUM>-<NUM> <TIME> info [inventory] stock updated sku=cw-<NUM> qty=<NUM> warehouse=wh-<NUM>,124



Rare Events:


,event_id,template,count
98,E0099,"{""ts"": ""<NUM>-<NUM>-15t08:<NUM>:<NUM>.811781z"", ""level"": ""error"", ""svc"": ""auth-service"", ""msg"": ""certificate pinning violation detected from client_ip=<IP>, possible mitm attack""}",1
88,E0064,^c,1
89,E0072,"<NUM>-<NUM>-<NUM> <TIME> error [inventory] cascade failure: warehouse sync triggered <NUM>,<NUM> duplicate sku entries in <NUM>.3s",1
90,E0077,???unexpected binary data??? \x00\x1f\x8b\x08,1
91,E0084,--- connection reset ---,1
97,E0098,<NUM>-<NUM>-15t08:<NUM>:<NUM>.019621z error kernel: kernel panic: unable to handle page fault at ffffffff81a00000 cpu#<NUM>,1
93,E0089,########## log restart ##########,1
94,E0092,"<NUM><PATH> +<NUM> | error | api-gateway | memory usage critical: <NUM>.<NUM>% of 16gb heap consumed, gc unable to free sufficient memory",1
95,E0094,<NUM>-<NUM>-<NUM> <TIME> warn [email-service] dns resolution intermittent for smtp.provider.com - <NUM> failures in last 60s,1
96,E0097,mar <NUM> <TIME> prod-web-<NUM> auth-service[<NUM>]: <warn> unusual login pattern detected: uid=<NUM> logged in from <NUM> countries in <NUM> hours,1


In [63]:
import re

def extract_severity(log):
    log = log.lower()
    
    if "error" in log:
        return "ERROR"
    elif "warn" in log:
        return "WARN"
    elif "info" in log:
        return "INFO"
    else:
        return "UNKNOWN"

df["severity"] = df["raw_log"].apply(extract_severity)

display(df[["raw_log", "severity"]].head(20))

,raw_log,severity
0,mar 15 06:00:03 prod-app-01 api-gateway[3330]: <info> health check passed upstream=auth-service,INFO
1,2026-03-15 06:00:03 info [api-gateway] request completed method=get path=/api/v2/users/3591 status=200 duration=264ms,INFO
2,2026-03-15 06:00:07 warn [inventory] sync delay from erp: last update 47 minutes ago,WARN
3,2026-03-15 06:00:09 info [inventory] stock updated sku=gw-7547 qty=328 warehouse=wh-03,INFO
4,2026-03-15t06:00:11.927231z error auth-service: mfa verification failed uid=2160 code=invalid,ERROR
5,2026-03-15 06:00:14 warn [api-gateway] request slow method=get path=/api/v2/search status=200 duration=2021ms,WARN
6,[06:00:17] i/auth-service: password reset requested uid=9085,UNKNOWN
7,mar 15 06:00:21 prod-app-01 inventory[4727]: <info> inventory sync completed source=erp records=2847 duration=12.4s,INFO
8,[06:00:25] i/email-service: email sent to=user677@company.io template=welcome msg_id=msg-739098,UNKNOWN
9,[06:00:28] i/api-gateway: request completed method=get path=/api/v2/products status=200 duration=49ms,UNKNOWN


In [64]:
severity_map = {
    "ERROR": 3,
    "WARN": 2,
    "INFO": 1,
    "UNKNOWN": 0
}

df["severity_score"] = df["severity"].map(severity_map)

In [ ]:
# Map severity score to readable label
def severity_label(avg):
    if avg >= 2.5:
        return "HIGH (Mostly ERROR)"
    elif avg >= 1.5:
        return "MEDIUM (WARN/ERROR mix)"
    elif avg >= 0.5:
        return "LOW (Mostly INFO)"
    else:
        return "INFO/UNKNOWN"

# Event level aggregation
event_severity = df.groupby("event_id").agg({
    "severity_score": "mean",
    "count": "first",
    "template": "first"
}).reset_index()

# Rename column
event_severity.rename(columns={"severity_score": "avg_severity"}, inplace=True)

# Add readable severity column
event_severity["severity_label"] = event_severity["avg_severity"].apply(severity_label)

# 🔥 Scrollable display
from IPython.display import display, HTML

display(HTML("""
<style>
.dataframe {
    max-height: 500px;
    overflow-y: scroll;
    display: block;
}
</style>
"""))

pd.set_option('display.max_colwidth', None)

display(event_severity.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
}))

,event_id,avg_severity,count,template,severity_label
0,E0001,0.911475,305,mar prod-app- api-gateway[]: health check passed upstream=auth-service,LOW (Mostly INFO)
1,E0002,1.394000,500,-- info [api-gateway] request completed method=get path= status= duration=264ms,LOW (Mostly INFO)
2,E0003,1.208955,134,-- warn [inventory] sync delay from erp: last update minutes ago,LOW (Mostly INFO)
3,E0004,1.185484,124,-- info [inventory] stock updated sku=gw- qty= warehouse=wh-,LOW (Mostly INFO)
4,E0005,1.411268,710,--15t06::.927231z error auth-service: mfa verification failed uid= code=invalid,LOW (Mostly INFO)
5,E0006,0.000000,13,[] i password reset requested uid=,INFO/UNKNOWN
6,E0007,0.854167,192,[] i email sent to=user677@company.io template=welcome msg_id=msg-,LOW (Mostly INFO)
7,E0008,0.000000,61,[] i request completed method=get path= status= duration=49ms,INFO/UNKNOWN
8,E0009,2.083601,311,"{""ts"": ""--15t06::.047373z"", ""level"": ""info"", ""svc"": ""api-gateway"", ""msg"": ""health check passed upstream=inventory""}",MEDIUM (WARN/ERROR mix)
9,E0010,0.882353,85,-- info [payment-svc] payment method added uid= type=card last4=,LOW (Mostly INFO)


In [66]:
# Normalize count (low count = more anomalous)
event_severity["count_score"] = 1 / event_severity["count"]

# Final anomaly score
event_severity["anomaly_score"] = (
    event_severity["avg_severity"] * 0.7 + 
    event_severity["count_score"] * 0.3
)

In [68]:
# 🔹 Define readable anomaly level
def anomaly_label(score):
    if score >= 2.0:
        return "CRITICAL 🔥 (Frequent errors or severe rare issue)"
    elif score >= 1.2:
        return "HIGH ⚠️ (Important issue, needs attention)"
    elif score >= 0.5:
        return "MEDIUM ⚡ (Moderate anomaly)"
    else:
        return "LOW ✅ (Mostly normal behavior)"

# 🔹 Add label column
event_severity["anomaly_label"] = event_severity["anomaly_score"].apply(anomaly_label)

# 🔹 Sort anomalies
anomalies = event_severity.sort_values(by="anomaly_score", ascending=False)

print("🚨 Top Anomalies:")

# 🔥 Scrollable display (NOT limited to head)
from IPython.display import display, HTML

display(HTML("""
<style>
.dataframe {
    max-height: 500px;
    overflow-y: auto;
    overflow-x: auto;
    display: block;
}
</style>
"""))

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

display(anomalies.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
}))

🚨 Top Anomalies:


,event_id,avg_severity,count,template,severity_label,count_score,anomaly_score,anomaly_label
98,E0099,3.000000,1,"{""ts"": ""--15t08::.811781z"", ""level"": ""error"", ""svc"": ""auth-service"", ""msg"": ""certificate pinning violation detected from client_ip=, possible mitm attack""}",HIGH (Mostly ERROR),1.000000,2.400000,CRITICAL 🔥 (Frequent errors or severe rare issue)
71,E0072,3.000000,1,"-- error [inventory] cascade failure: warehouse sync triggered , duplicate sku entries in .3s",HIGH (Mostly ERROR),1.000000,2.400000,CRITICAL 🔥 (Frequent errors or severe rare issue)
97,E0098,3.000000,1,--15t08::.019621z error kernel: kernel panic: unable to handle page fault at ffffffff81a00000 cpu#,HIGH (Mostly ERROR),1.000000,2.400000,CRITICAL 🔥 (Frequent errors or severe rare issue)
91,E0092,3.000000,1,"+ | error | api-gateway | memory usage critical: .% of 16gb heap consumed, gc unable to free sufficient memory",HIGH (Mostly ERROR),1.000000,2.400000,CRITICAL 🔥 (Frequent errors or severe rare issue)
94,E0095,3.000000,2,"{""ts"": ""--15t08::.133109z"", ""level"": ""error"", ""svc"": ""api-gateway"", ""msg"": ""tls handshake failed client_ip= error=certificate_expired"", ""trace_id"": ""trace-""}",HIGH (Mostly ERROR),0.500000,2.250000,CRITICAL 🔥 (Frequent errors or severe rare issue)
90,E0091,3.000000,2,"+ | error | payment-svc | data integrity alert: checksum mismatch on transaction log segment 0x4f2a, possible corruption",HIGH (Mostly ERROR),0.500000,2.250000,CRITICAL 🔥 (Frequent errors or severe rare issue)
59,E0060,3.000000,3,"{""ts"": ""--15t06::.984644z"", ""level"": ""error"", ""svc"": ""payment-svc"", ""msg"": ""payment declined order_id=ord- reason=insufficient_funds""}",HIGH (Mostly ERROR),0.333333,2.200000,CRITICAL 🔥 (Frequent errors or severe rare issue)
86,E0087,3.000000,5,"{""ts"": ""--15t07::.093450z"", ""level"": ""error"", ""svc"": ""scheduler"", ""msg"": ""scheduler heartbeat missed, possible zombie process pid=""}",HIGH (Mostly ERROR),0.200000,2.160000,CRITICAL 🔥 (Frequent errors or severe rare issue)
58,E0059,3.000000,7,panic: runtime error: index out of range [] with length,HIGH (Mostly ERROR),0.142857,2.142857,CRITICAL 🔥 (Frequent errors or severe rare issue)
81,E0082,3.000000,7,"{""ts"": ""--15t07::.581463z"", ""level"": ""error"", ""svc"": ""auth-service"", ""msg"": ""token signing key rotation failed: keystore locked""}",HIGH (Mostly ERROR),0.142857,2.142857,CRITICAL 🔥 (Frequent errors or severe rare issue)
